In [12]:
import sys
import numpy as np
import pandas as pd

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

pd.set_option('display.max_columns', None)

In [13]:
dtype = np.float64
intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCh38/psix_annotation/intron_file.tab.gz"
intron_table = read_intron_file(intron_file)

In [3]:
corr_df = pd.read_csv(f"data/corrs/GTEx_frontal_cortex_TPM_OR_lmFit_residuals_mergeParam0.85_top_MO_Qval_modules_exon_corr.csv", index_col=0)
top_qval_mods_df = pd.read_csv("data/enrichments/GTEx_frontal_cortex_TPM_OR_lmFit_residuals_mergeParam0.85_top_MO_Qval_modules.csv")
expr_tpm = pd.read_csv("data/GTEx_frontal_cortex_TPM_OR_lmFit_residuals.csv", index_col=0)

In [4]:
ctypes = top_qval_mods_df['Cell_type'].loc[top_qval_mods_df['Qval'] < 1e-20].values

In [5]:
ctypes

array(['YANG_PFC_2021_OLIGODENDROCYTE', 'YANG_PFC_2021_ASTROCYTE',
       'YANG_PFC_2021_ENDOTHELIAL', 'YANG_PFC_2021_MICROGLIA',
       'YANG_PFC_2021_L2_3_EXCITATORY_NEURON',
       'YANG_PFC_2021_NRGN_NEURON', 'YANG_PFC_2021_VIP_INTERNEURON',
       'BARRES_ASTROCYTES', 'HICKMAN_MICROGLIA_SENSOME_99',
       'ZEISEL_INTERNEURON', 'BARRES_OLIGODENDROCYTES', 'ZEISEL_MURAL',
       'Mukamel_InhibitoryNeurons',
       'BAKKEN_2019_SST_CHODL_GABAERGIC_DE_GABA_CLUSTERS'], dtype=object)

In [6]:
mean_expr = expr_tpm.mean(axis=1)
mean_expr.name = "Mean_expr"

# Add mean expression

col_order = ["Gene", "Exon", "Mean_expr"] + ctypes.tolist()

corr_expr_df = (
    corr_df.reset_index(names="Exon")
          .merge(mean_expr, left_on="Gene", right_index=True, how="left")
)
corr_expr_df = corr_expr_df[col_order]

In [21]:
idx = 4
print("Working cell type:", ctypes[idx])

ctype_row = top_qval_mods_df['Cell_type'] == ctypes[idx]
mod = top_qval_mods_df.loc[ctype_row, 'Module'].values[0]
print("Working module:", mod)
kme_df = pd.read_csv(top_qval_mods_df.loc[ctype_row, 'kME_path'].values[0])
kme_cols = [kme_df.columns[0], *kme_df.columns[kme_df.columns.str.contains("TopModPosBC")],  f"kME{mod}"]
kme_mod_df = kme_df[kme_cols]

corr_mod_df = kme_mod_df.merge(corr_expr_df, left_on="Gene", right_on="Gene")

# # Filter for genes NOT in the working cell type module:
# mask = ~corr_mod_df.iloc[:, 1].isin([mod])
# corr_mod_df[mask].sort_values(ctypes[idx], ascending=False).head(20)

corr_mod_df.sort_values(ctypes[idx], ascending=False).head(10)

Working cell type: YANG_PFC_2021_L2_3_EXCITATORY_NEURON
Working module: blue


,Gene,TopModPosBC_6.7e-07,kMEblue,Exon,Mean_expr,YANG_PFC_2021_OLIGODENDROCYTE,YANG_PFC_2021_ASTROCYTE,YANG_PFC_2021_ENDOTHELIAL,YANG_PFC_2021_MICROGLIA,YANG_PFC_2021_L2_3_EXCITATORY_NEURON,YANG_PFC_2021_NRGN_NEURON,YANG_PFC_2021_VIP_INTERNEURON,BARRES_ASTROCYTES,HICKMAN_MICROGLIA_SENSOME_99,ZEISEL_INTERNEURON,BARRES_OLIGODENDROCYTES,ZEISEL_MURAL,Mukamel_InhibitoryNeurons,BAKKEN_2019_SST_CHODL_GABAERGIC_DE_GABA_CLUSTERS
8634,DLX6-AS1,blue,0.666994,ENSG00000231764_other_8,5.273184,-0.177538,-0.198585,-0.112975,-0.004348,0.357092,0.300706,0.357092,-0.193593,-0.010353,0.343147,-0.207629,0.010836,0.351351,0.243084
29399,SLAIN1,turquoise,-0.543944,ENSG00000139737_ProteinCoding_1,48.765993,-0.235947,-0.130677,-0.129959,-0.036617,0.350530,0.268412,0.350530,-0.129296,-0.036261,0.245048,-0.256330,-0.055086,0.242346,0.221986
8630,DLX6-AS1,blue,0.666994,ENSG00000231764_other_4,5.273184,-0.201433,-0.185240,-0.116912,0.032722,0.346297,0.289313,0.346297,-0.180662,0.028791,0.300434,-0.232887,-0.005550,0.303125,0.244955
27037,RELN,blue,0.636274,ENSG00000189056_ProteinCoding_2,6.188311,-0.301094,0.011994,-0.140058,-0.010456,0.343332,0.181702,0.343332,0.017099,-0.015744,0.325883,-0.293509,-0.017378,0.319610,0.148841
27009,REEP3,turquoise,-0.618326,ENSG00000165476_NMD_1,9.836251,-0.126897,-0.192174,-0.208595,-0.122724,0.329992,0.290401,0.329992,-0.190239,-0.125648,0.210609,-0.160780,-0.128188,0.187794,0.242298
27636,ROBO1,NaN,0.205218,ENSG00000169855_ProteinCoding_1,11.117099,-0.198172,-0.085095,-0.180924,-0.058529,0.324179,0.208994,0.324179,-0.085681,-0.064267,0.261852,-0.224044,-0.123907,0.253553,0.165732
36169,WNK1,turquoise,-0.252600,ENSG00000060237_ProteinCoding_6,76.687963,-0.204421,-0.082508,-0.070076,-0.039556,0.322821,0.270196,0.322821,-0.078191,-0.040559,0.166248,-0.216381,-0.033376,0.167673,0.222199
23018,PBX1,blue,0.570329,ENSG00000185630_ProteinCoding_3,27.463300,-0.056290,-0.329969,-0.206766,-0.042318,0.317963,0.371185,0.317963,-0.327014,-0.046269,0.274757,-0.101290,-0.075231,0.271055,0.344626
8908,DNM2,turquoise,-0.698872,ENSG00000079805_ProteinCoding_1,32.539696,-0.159810,-0.127087,-0.178482,-0.152535,0.317521,0.223927,0.317521,-0.127121,-0.154177,0.279432,-0.166184,-0.105401,0.263815,0.182514
8629,DLX6-AS1,blue,0.666994,ENSG00000231764_other_3,5.273184,-0.166205,-0.161709,-0.162385,0.025627,0.317231,0.282414,0.317231,-0.158869,0.024228,0.290837,-0.200428,-0.034618,0.301864,0.240154


In [22]:
prefix = "ENSG00000060237_ProteinCoding_6"
rows = intron_table.loc[intron_table.index.str.startswith(prefix)]
rows

,intron
ENSG00000060237_ProteinCoding_6_I1,chr12:878362-880720:+
ENSG00000060237_ProteinCoding_6_I2,chr12:881000-881691:+
ENSG00000060237_ProteinCoding_6_SE,chr12:878362-881691:+


In [ ]:
50065578-

-55